# ПРЕДПОДГОТОВКА ДАННЫХ

### Цели и задачи проекта

Подготовить данные о видеоиграх к дальнейшему исследовательскому анализу: привести структуру и типы данных к рабочему состоянию, обработать пропуски и дубликаты, нормализовать категориальные признаки и сформировать аналитический срез за 2000-2013 годы.


### Описание данных

В исходном датасете содержится информация о компьютерных и консольных играх: название, платформа, год выпуска, жанр, региональные продажи, а также пользовательские и критические оценки. Данные собраны из открытых источников и охватывают период с 1980 по 2016 годы.

### Содержимое проекта

1. Загрузка данных и первичный обзор.
2. Проверка ошибок и предобработка:
   - названия столбцов;
   - типы данных;
   - пропуски;
   - явные и неявные дубликаты.
3. Фильтрация данных по периоду 2000-2013.
4. Категоризация по пользовательским и критическим оценкам.
5. Итоговый вывод.


## 1. Загрузка данных и знакомство с ними

Загрузим необходимые библиотеки Python и данные датасета `/datasets/new_games.csv`.


На этом этапе выполнен первичный обзор датасета: проверены структура, типы данных, первые и последние строки, а также базовые статистические показатели.


In [ ]:
import pandas as pd #импортируем библиотеку
from pathlib import Path

data_candidates = [Path('datasets/new_games.csv'), Path('../datasets/new_games.csv')]
for data_path in data_candidates:
    if data_path.exists():
        dataset = pd.read_csv(data_path) #загружаем датасет
        break
else:
    checked_paths = ', '.join(str(path) for path in data_candidates)
    raise FileNotFoundError(f"Не найден файл датасета. Проверены пути: {checked_paths}")


In [2]:
dataset.info() #общая информация о датасете
dataset.describe() #основные стат. показатели для числовых столбцов

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  str    
 1   Platform         16956 non-null  str    
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  str    
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  str    
 6   JP sales         16956 non-null  str    
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  str    
 10  Rating           10085 non-null  str    
dtypes: float64(4), str(7)
memory usage: 1.4 MB


,Year of Release,NA sales,Other sales,Critic Score
count,16681.000000,16956.000000,16956.000000,8242.000000
mean,2006.485522,0.262023,0.047087,68.926717
std,5.873102,0.808654,0.185577,13.944565
min,1980.000000,0.000000,0.000000,13.000000
25%,2003.000000,0.000000,0.000000,60.000000
50%,2007.000000,0.080000,0.010000,71.000000
75%,2010.000000,0.240000,0.030000,79.000000
max,2016.000000,41.360000,10.570000,98.000000


1. В датасете 11 столбцов и 16956 строк; часть числовых признаков загружена в строковом типе (`str`/`object` в зависимости от версии pandas).
2. Обнаружены пропуски в 6 столбцах.
3. По сводной статистике заметны существенные расхождения между средними и медианами, что указывает на асимметрию распределений и наличие выбросов.


In [3]:
dataset.head() # первые 5 строк

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [4]:
dataset.tail() # последние 5 строк

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
16951,Samurai Warriors: Sanada Maru,PS3,2016.0,Action,0.00,0.0,0.01,0.0,NaN,NaN,NaN
16952,LMA Manager 2007,X360,2006.0,Sports,0.00,0.01,0.0,0.0,NaN,NaN,NaN
16953,Haitaka no Psychedelica,PSV,2016.0,Adventure,0.00,0.0,0.01,0.0,NaN,NaN,NaN
16954,Spirits & Spells,GBA,2003.0,Platform,0.01,0.0,0.0,0.0,NaN,NaN,NaN
16955,Winning Post 8 2016,PSV,2016.0,Simulation,0.00,0.0,0.01,0.0,NaN,NaN,NaN


Первичный просмотр подтвердил соответствие структуры описанию данных.
Ключевые проблемы для предобработки: некорректные типы в части столбцов и значительная доля пропусков в `critic_score`, `user_score` и `rating`.
Для целей исследования это допустимо: отсутствующие оценки и возрастные рейтинги не блокируют анализ динамики рынка.


---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

Выведем на экран названия всех столбцов датафрейма и проверим их стиль написания.
Приведем все столбцы к стилю snake case.

In [5]:
dataset.columns # проверим названия столбцов

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='str')

In [6]:
dataset.columns = dataset.columns.str.lower().str.replace(' ', '_', regex=False) # приведем все к нижнему регистру и snake_case
dataset.columns


Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='str')

### 2.2. Типы данных

Проверим соответствие типов данных содержимому столбцов.
При необходимости выполним преобразование к корректным типам.


In [7]:
dataset.dtypes

name                   str
platform               str
year_of_release    float64
genre                  str
na_sales           float64
eu_sales               str
jp_sales               str
other_sales        float64
critic_score       float64
user_score             str
rating                 str
dtype: object

In [8]:
# Выводим все уникальные значения без сокращения (как Python-списки)
print(dataset['eu_sales'].unique().tolist()) # смотрим, что в eu_sales нечислового
print(dataset['jp_sales'].unique().tolist()) # смотрим jp_sales
print(dataset['user_score'].unique().tolist()) # смотрим user_score
print(dataset['year_of_release'].unique().tolist())
print('Количество пропусков:', dataset['year_of_release'].isna().sum()) # смотрим столбец year_of_release


['28.96', '3.58', '12.76', '10.93', '8.89', '2.26', '9.14', '9.18', '6.94', '0.63', '10.95', '7.47', '6.18', '8.03', '4.89', '8.49', '9.09', '0.4', '3.75', '9.2', '4.46', '2.71', '3.44', '5.14', '5.49', '3.9', '5.35', '3.17', '5.09', '4.24', '5.04', '5.86', '3.68', '4.19', '5.73', '3.59', '4.51', '2.55', '4.02', '4.37', '6.31', '3.45', '2.81', '2.85', '3.49', '0.01', '3.35', '2.04', '3.07', '3.87', '3.0', '4.82', '3.64', '2.15', '3.69', '2.65', '2.56', '3.11', '3.14', '1.94', '1.95', '2.47', '2.28', '3.42', '3.63', '2.36', '1.71', '1.85', '2.79', '1.24', '6.12', '1.53', '3.47', '2.24', '5.01', '2.01', '1.72', '2.07', '6.42', '3.86', '0.45', '3.48', '1.89', '5.75', '2.17', '1.37', '2.35', '1.18', '2.11', '1.88', '2.83', '2.99', '2.89', '3.27', '2.22', '2.14', '1.45', '1.75', '1.04', '1.77', '3.02', '2.75', '2.16', '1.9', '2.59', '2.2', '4.3', '0.93', '2.53', '2.52', '1.79', '1.3', '2.6', '1.58', '1.2', '1.56', '1.34', '1.26', '0.83', '6.21', '2.8', '1.59', '1.73', '4.33', '1.83', '0.0',

Вывод:

При проверке уникальных значений выявлено следующее:
1. В `eu_sales` и `jp_sales` присутствуют текстовые значения `unknown`, поэтому столбцы были загружены в строковом типе (`str`/`object` в зависимости от версии pandas).
2. В `user_score` присутствуют `NaN` и `tbd`; из-за `tbd` столбец также определяется как строковый (`str`/`object` в зависимости от версии pandas).
3. `year_of_release` определен как `float64`, так как содержит пропуски. Для корректной работы этот столбец целесообразно перевести в тип `Int64`, который поддерживает `NaN`.


In [9]:
dataset['year_of_release'] = pd.to_numeric(dataset['year_of_release'], errors='coerce').astype('Int64') #приведем столбец к значению Int64 т.к. он поддерживает пропуски в целочисленных столбцах
dataset['eu_sales'] = pd.to_numeric(dataset['eu_sales'], errors='coerce').astype('float64') #тут и далее приведем столбцы к типу float64, заменив строковые значения на NaN
dataset['jp_sales'] = pd.to_numeric(dataset['jp_sales'], errors='coerce').astype('float64')
dataset['user_score'] = pd.to_numeric(dataset['user_score'], errors='coerce').astype('float64')


In [10]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  str    
 1   platform         16956 non-null  str    
 2   year_of_release  16681 non-null  Int64  
 3   genre            16954 non-null  str    
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float64
 10  rating           10085 non-null  str    
dtypes: Int64(1), float64(6), str(4)
memory usage: 1.4 MB


In [11]:
dataset.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006,Sports,41.36,28.96,3.77,8.45,76.0,8.0,E
1,Super Mario Bros.,NES,1985,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009,Sports,15.61,10.93,3.28,2.95,80.0,8.0,E
4,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Вывод:

Столбцы приведены к формату, пригодному для дальнейшей работы:
1. Для `eu_sales`, `jp_sales`, `user_score` текстовые значения преобразованы в `NaN`, затем выполнено приведение к `float64`.
2. `year_of_release` приведен к типу `Int64`, который является целочисленным и поддерживает пропуски.

Причина исходных несоответствий - смешение чисел и текста (`unknown`, `tbd`) и наличие пропусков в годе выпуска.


### 2.3. Наличие пропусков в данных

Посчитаем количество пропусков в каждом столбце в абсолютных и относительных значениях.


In [12]:
dataset.isna().sum() # считаем количество пропусков в каждом столбце в абсолютных значениях.

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

In [13]:
dataset.isna().mean()*100 # считаем количество пропусков в каждом столбце в относительных значениях.

name                0.011795
platform            0.000000
year_of_release     1.621845
genre               0.011795
na_sales            0.000000
eu_sales            0.035386
jp_sales            0.023590
other_sales         0.000000
critic_score       51.391838
user_score         54.659118
rating             40.522529
dtype: float64

- `name` и `genre` - по 2 пропуска (0.01%): это критичные поля, поэтому строки удалены.
- `year_of_release` - 275 пропусков (1.62%): столбец ключевой для среза 2000-2013, строки с пропусками удалены.
- `eu_sales` и `jp_sales` - 6 и 4 пропуска (менее 0.05%): пропуски заполнены средними значениями по платформе и году.
- `critic_score` (51.39%), `user_score` (54.66%), `rating` (40.52%) - высокая доля пропусков; значения оставлены без заполнения, чтобы не вносить систематическое искажение.


In [15]:
dataset_drop = dataset.dropna(subset = ['name', 'year_of_release', 'genre']).copy() #удаляем значения из колонок 'name', 'year_of_release', 'genre'

In [16]:
mean_eu_sales = dataset.groupby(['platform', 'year_of_release'])['eu_sales'].mean() # считаем среднее по платформе и году
def fill_eu_sales(row): # определяем функцию
    if pd.isna(row['eu_sales']): # ищем пропуски
        plat = row['platform'] # определяем значения года и платформы конкретной строки
        year = row['year_of_release']
        if (plat, year) in mean_eu_sales.index: # проверяем, есть ли совпадение по платформе и году
            return mean_eu_sales[(plat, year)] # если есть, подставляем среднее значение
        else:
            return row['eu_sales']  # если нет, оставим как есть
    return row['eu_sales'] # если пропуска не было, осавляем текущее значение
dataset_drop['eu_sales'] = dataset_drop.apply(fill_eu_sales, axis=1) # применяем функцию к датасету построчно

In [17]:
mean_jp_sales = dataset.groupby(['platform', 'year_of_release'])['jp_sales'].mean() # аналогично расчету для eu_sales
def fill_jp_sales(row):
    if pd.isna(row['jp_sales']):
        plat = row['platform']
        year = row['year_of_release']
        if (plat, year) in mean_jp_sales.index:
            return mean_jp_sales[(plat, year)]
        else:
            return row['jp_sales']
    return row['jp_sales']
dataset_drop['jp_sales'] = dataset_drop.apply(fill_jp_sales, axis=1)


In [18]:
dataset_drop.isna().sum() #проверяем на пропуски после обработки

name                  0
platform              0
year_of_release       0
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8594
user_score         9121
rating             6778
dtype: int64

Пропуски в ключевых полях (`name`, `year_of_release`, `genre`) удалены.
Пропуски в `eu_sales` и `jp_sales` заполнены средними значениями по связке "платформа + год".
Оставшиеся пропуски в `critic_score`, `user_score`, `rating` сохранены без изменений из-за их высокой доли.


### 2.4. Явные и неявные дубликаты в данных

Проверены уникальные значения категориальных полей (`genre`, `platform`, `rating`, `year_of_release`) и выполнена нормализация регистра.
Далее проведена проверка явных дубликатов как по всем столбцам, так и по ключевым полям (`name`, `platform`, `year_of_release`, `genre`).


In [19]:
dataset_drop.nunique()

name               11426
platform              31
year_of_release       37
genre                 24
na_sales             401
eu_sales             313
jp_sales             248
other_sales          155
critic_score          81
user_score            95
rating                 8
dtype: int64

In [20]:
print(dataset_drop['platform'].unique())
print(dataset_drop['genre'].unique())
print(dataset_drop['year_of_release'].unique())
print(dataset_drop['rating'].unique())

<StringArray>
[ 'Wii',  'NES',   'GB',   'DS', 'X360',  'PS3',  'PS2', 'SNES',  'GBA',
  'PS4',  '3DS',  'N64',   'PS',   'XB',   'PC', '2600',  'PSP', 'XOne',
 'WiiU',   'GC',  'GEN',   'DC',  'PSV',  'SAT',  'SCD',   'WS',   'NG',
 'TG16',  '3DO',   'GG', 'PCFX']
Length: 31, dtype: str
<StringArray>
[      'Sports',     'Platform',       'Racing', 'Role-Playing',
       'Puzzle',         'Misc',      'Shooter',   'Simulation',
       'Action',     'Fighting',    'Adventure',     'Strategy',
         'MISC', 'ROLE-PLAYING',       'RACING',       'ACTION',
      'SHOOTER',     'FIGHTING',       'SPORTS',     'PLATFORM',
    'ADVENTURE',   'SIMULATION',       'PUZZLE',     'STRATEGY']
Length: 24, dtype: str
<IntegerArray>
[2006, 1985, 2008, 2009, 1996, 1989, 1984, 2005, 1999, 2007, 2010, 2013, 2004,
 1990, 1988, 2002, 2001, 2011, 1998, 2015, 2012, 2014, 1992, 1997, 1993, 1994,
 1982, 2016, 2003, 1986, 2000, 1995, 1991, 1981, 1987, 1980, 1983]
Length: 37, dtype: Int64
<StringArray>
['E',

In [21]:
dataset_drop['genre'] = dataset_drop['genre'].str.lower() # genre к нижнему регистру
dataset_drop['rating'] = dataset_drop['rating'].str.upper() # rating к верхнему регистру


In [22]:
print(dataset_drop['rating'].unique())
print(dataset_drop['genre'].unique())
dataset_drop.nunique()

<StringArray>
['E', nan, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']
Length: 9, dtype: str
<StringArray>
[      'sports',     'platform',       'racing', 'role-playing',
       'puzzle',         'misc',      'shooter',   'simulation',
       'action',     'fighting',    'adventure',     'strategy']
Length: 12, dtype: str


name               11426
platform              31
year_of_release       37
genre                 12
na_sales             401
eu_sales             313
jp_sales             248
other_sales          155
critic_score          81
user_score            95
rating                 8
dtype: int64

После нормализации регистра количество уникальных жанров сократилось с 24 до 12, что подтверждает наличие неявных дубликатов по написанию.
Также в `rating` обнаружено значение `K-A` - это устаревшая категория ESRB, поэтому признак сохранен как корректный исторический вариант.


In [23]:
dataset_drop.duplicated().sum() # проверяем на явные дубликаты по всем полям

np.int64(235)

In [24]:
dataset_drop2 = dataset_drop.drop_duplicates(keep='first') # удаляем явные дубликаты, кроме первой строчки.
dataset_drop2.duplicated().sum() # проверяем

np.int64(0)

In [25]:
dataset_drop2.duplicated(subset=['name', 'platform', 'year_of_release', 'genre']).sum()# проверяем на явные дубликаты по ключевым полям

np.int64(1)

In [26]:
dataset_drop3 = dataset_drop2.drop_duplicates(subset=['name', 'platform', 'year_of_release', 'genre'], keep='first') # удаляем дубликаты по ключевым полям
dataset_drop3.duplicated(subset=['name', 'platform', 'year_of_release', 'genre']).sum() # проверяем


np.int64(0)

In [27]:
dataset_drop3.info()

<class 'pandas.DataFrame'>
Index: 16443 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16443 non-null  str    
 1   platform         16443 non-null  str    
 2   year_of_release  16443 non-null  Int64  
 3   genre            16443 non-null  str    
 4   na_sales         16443 non-null  float64
 5   eu_sales         16443 non-null  float64
 6   jp_sales         16443 non-null  float64
 7   other_sales      16443 non-null  float64
 8   critic_score     7982 non-null   float64
 9   user_score       7462 non-null   float64
 10  rating           9767 non-null   str    
dtypes: Int64(1), float64(6), str(4)
memory usage: 1.5 MB


Обнаружено 235 полных дубликатов и 1 дубликат по ключевым полям (`name`, `platform`, `year_of_release`, `genre`).
Дубликаты удалены с сохранением первой записи.
Итоговый объем датасета после очистки: 16443 строки.


In [28]:
start = len(dataset)
finish = len(dataset_drop3)
removed_abs = start - finish
removed_otn = round((removed_abs / start) * 100, 2)

print(f'Удалено строк: {removed_abs} ({removed_otn}%)')

Удалено строк: 513 (3.03%)


**Предобработка данных**

- Исходный объем данных: 16956 строк.
- После очистки и обработки: 16443 строки.
- Удалено: 513 строк (3.03%).

**Что было сделано:**

1. Названия столбцов приведены к snake_case.
2. Типы данных приведены к рабочему формату (`year_of_release` - `Int64`; продажи и оценки - `float64`).
3. Удалены строки с пропусками в `name`, `year_of_release`, `genre`.
4. Пропуски в `eu_sales` и `jp_sales` заполнены средними значениями по платформе и году.
5. Пропуски в `critic_score`, `user_score`, `rating` оставлены без заполнения из-за их высокой доли.
6. Категориальные значения нормализованы (`genre` - нижний регистр, `rating` - верхний).
7. Удалены 235 полных дубликатов и 1 дубликат по ключевым полям.

Данные приведены к рабочему виду и готовы к дальнейшему анализу.


---

## 3. Фильтрация данных

Для анализа динамики продаж сформирован срез за период 2000-2013 годов включительно. Результат сохранен в отдельном датафрейме `df_actual`.


In [29]:
df_actual = dataset_drop3.query('year_of_release >= 2000 and year_of_release <= 2013') # фильтрация по целевому периоду
df_actual.info()


<class 'pandas.DataFrame'>
Index: 12780 entries, 0 to 16954
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             12780 non-null  str    
 1   platform         12780 non-null  str    
 2   year_of_release  12780 non-null  Int64  
 3   genre            12780 non-null  str    
 4   na_sales         12780 non-null  float64
 5   eu_sales         12780 non-null  float64
 6   jp_sales         12780 non-null  float64
 7   other_sales      12780 non-null  float64
 8   critic_score     7168 non-null   float64
 9   user_score       6482 non-null   float64
 10  rating           8722 non-null   str    
dtypes: Int64(1), float64(6), str(4)
memory usage: 1.2 MB


Из очищенного датасета (16443 строки) отобраны игры, выпущенные в период 2000-2013 годов включительно.
Итоговый срез `df_actual` содержит 12780 строк.


---

## 4. Категоризация данных

Для дальнейшего анализа добавлены категории пользовательских и критических оценок на основе заданных интервальных правил.


In [30]:
df_actual = df_actual.copy() # работаем с независимой копией
df_actual['user_category'] = 'нет_оценки' # категория по умолчанию
df_actual.loc[(df_actual['user_score'] >= 0) & (df_actual['user_score'] < 3), 'user_category'] = 'низкая_оценка' # низкая оценка: от 0 до 3, правая граница не включается
df_actual.loc[(df_actual['user_score'] >= 3) & (df_actual['user_score'] < 8), 'user_category'] = 'средняя_оценка' # средняя оценка: от 3 до 8, правая граница не включается
df_actual.loc[(df_actual['user_score'] >= 8) & (df_actual['user_score'] <= 10), 'user_category'] = 'высокая_оценка' # высокая оценка: от 8 до 10 включительно

df_actual['user_category'].value_counts(dropna=False) # распределение по категориям


user_category
нет_оценки        6298
средняя_оценка    4080
высокая_оценка    2286
низкая_оценка      116
Name: count, dtype: int64

Для оценок критиков использованы интервалы: высокая (80-100), средняя [30-80), низкая [0-30).
Записи без значения `critic_score` отнесены к категории `нет_оценки`.


In [31]:
df_actual = df_actual.copy() # работаем с независимой копией
df_actual['critic_category'] = 'нет_оценки'
df_actual.loc[(df_actual['critic_score'] >= 0) & (df_actual['critic_score'] < 30), 'critic_category'] = 'низкая_оценка'
df_actual.loc[(df_actual['critic_score'] >= 30) & (df_actual['critic_score'] < 80), 'critic_category'] = 'средняя_оценка'
df_actual.loc[(df_actual['critic_score'] >= 80) & (df_actual['critic_score'] <= 100), 'critic_category'] = 'высокая_оценка'
df_actual['critic_category'].value_counts(dropna=False)


critic_category
нет_оценки        5612
средняя_оценка    5422
высокая_оценка    1691
низкая_оценка       55
Name: count, dtype: int64

После категоризации видно, что наибольшая доля записей относится к категории `нет_оценки` как у пользователей, так и у критиков.
Среди наблюдений с доступной оценкой преобладает категория `средняя_оценка`.


Далее выделим топ-7 платформ по количеству выпущенных игр за выбранный период.


In [ ]:
df_actual.groupby('platform')['name'].count().sort_values(ascending=False).head(7)

platform
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1086
GBA      811
Name: name, dtype: int64

Определен топ-7 платформ по количеству релизов в 2000-2013 гг.: PS2, DS, Wii, PSP, X360, PS3, GBA.


## 5. Итоговый вывод

В проекте выполнена предобработка датасета видеоигр и сформирован аналитический срез для периода 2000-2013.

Что сделано:

- Названия столбцов унифицированы в формате snake_case.
- Типы данных приведены к корректным (`year_of_release` - `Int64`, числовые метрики - `float64`).
- Удалены пропуски в ключевых полях (`name`, `year_of_release`, `genre`).
- Пропуски в `eu_sales` и `jp_sales` заполнены средними по платформе и году.
- Пропуски в `critic_score`, `user_score`, `rating` сохранены без заполнения.
- Нормализованы категориальные значения (`genre`, `rating`).
- Удалены 235 полных дубликатов и 1 дубликат по ключевым полям.

Итог:

- Очищенный датасет: 16443 строки.
- Срез `df_actual` за 2000-2013: 12780 строк.
- Добавлены поля `user_category` и `critic_category`.
- Топ-7 платформ по числу релизов: PS2, DS, Wii, PSP, X360, PS3, GBA.
